In [1]:
import pandas as pd
import os

In [2]:
sales = pd.read_csv('../data/raw/car_prices.csv')
print(sales.shape)
sales.head()

(558837, 16)


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


In [3]:
fuel = pd.read_csv('../data/raw/PET_PRI_GND_DCUS_NUS_W.csv')
print(fuel.shape)
fuel.head()

(1361, 14)


,Date,A1,A2,A3,R1,R2,R3,M1,M2,M3,P1,P2,P3,D1
0,01/02/1995,1.127,1.104,1.231,1.079,1.063,1.167,1.170,1.159,1.298,1.272,1.250,1.386,1.104
1,01/09/1995,1.134,1.111,1.232,1.086,1.070,1.169,1.177,1.164,1.300,1.279,1.256,1.387,1.102
2,01/16/1995,1.126,1.102,1.231,1.078,1.062,1.169,1.168,1.155,1.299,1.271,1.249,1.385,1.100
3,01/23/1995,1.132,1.110,1.226,1.083,1.068,1.165,1.177,1.165,1.296,1.277,1.256,1.378,1.095
4,01/30/1995,1.131,1.109,1.221,1.083,1.068,1.162,1.176,1.163,1.291,1.275,1.255,1.370,1.090


In [4]:
temp = pd.read_csv('../data/raw/archive/temperature.csv')
print(temp.shape)
temp.head()

(45253, 37)


,datetime,Vancouver,Portland,San Francisco,Seattle,Los Angeles,San Diego,Las Vegas,Phoenix,Albuquerque,...,Philadelphia,New York,Montreal,Boston,Beersheba,Tel Aviv District,Eilat,Haifa,Nahariyya,Jerusalem
0,2012-10-01 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,309.100000,NaN,NaN,NaN
1,2012-10-01 13:00:00,284.630000,282.080000,289.480000,281.800000,291.870000,291.530000,293.410000,296.600000,285.120000,...,285.630000,288.220000,285.830000,287.170000,307.590000,305.470000,310.580000,304.4,304.4,303.5
2,2012-10-01 14:00:00,284.629041,282.083252,289.474993,281.797217,291.868186,291.533501,293.403141,296.608509,285.154558,...,285.663208,288.247676,285.834650,287.186092,307.590000,304.310000,310.495769,304.4,304.4,303.5
3,2012-10-01 15:00:00,284.626998,282.091866,289.460618,281.789833,291.862844,291.543355,293.392177,296.631487,285.233952,...,285.756824,288.326940,285.847790,287.231672,307.391513,304.281841,310.411538,304.4,304.4,303.5
4,2012-10-01 16:00:00,284.624955,282.100481,289.446243,281.782449,291.857503,291.553209,293.381213,296.654466,285.313345,...,285.850440,288.406203,285.860929,287.277251,307.145200,304.238015,310.327308,304.4,304.4,303.5


In [5]:
print("Sales missing values:")
print(sales.isnull().sum())

Sales missing values:
year                0
make            10301
model           10399
trim            10651
body            13195
transmission    65352
vin                 4
state               0
condition       11820
odometer           94
color             749
interior          749
seller              0
mmr                38
sellingprice       12
saledate           12
dtype: int64


In [6]:
# Fix the date column (it's messy right now)
sales['saledate'] = pd.to_datetime(sales['saledate'], utc=True, errors='coerce')

# Extract useful time parts
sales['year_sold'] = sales['saledate'].dt.year
sales['month'] = sales['saledate'].dt.month

# Add season column
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

sales['season'] = sales['month'].apply(get_season)

# Drop rows missing critical columns
sales_clean = sales.dropna(subset=['sellingprice', 'saledate', 'body'])

print(sales_clean.shape)
sales_clean[['saledate','year_sold','month','season','body','sellingprice']].head()

/var/folders/dl/mldrhyqj2kv75r51k30p7w_00000gn/T/ipykernel_9727/2198320795.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales['saledate'] = pd.to_datetime(sales['saledate'], utc=True, errors='coerce')


(545604, 19)


,saledate,year_sold,month,season,body,sellingprice
0,2014-12-16 04:30:00+00:00,2014.0,12.0,Winter,SUV,21500.0
1,2014-12-16 04:30:00+00:00,2014.0,12.0,Winter,SUV,21500.0
2,2015-01-14 20:30:00+00:00,2015.0,1.0,Winter,Sedan,30000.0
3,2015-01-28 20:30:00+00:00,2015.0,1.0,Winter,Sedan,27750.0
4,2014-12-18 04:30:00+00:00,2014.0,12.0,Winter,Sedan,67000.0


In [7]:
# Fix date
fuel['Date'] = pd.to_datetime(fuel['Date'])

# Keep only date and regular gas price (A1 = regular unleaded)
fuel_clean = fuel[['Date', 'A1']].rename(columns={'A1': 'gas_price'})

# Add month and year for merging later
fuel_clean['month'] = fuel_clean['Date'].dt.month
fuel_clean['year_sold'] = fuel_clean['Date'].dt.year

# Get monthly average (sales data is monthly, fuel is weekly)
fuel_monthly = fuel_clean.groupby(['year_sold','month'])['gas_price'].mean().reset_index()

print(fuel_monthly.shape)
fuel_monthly.head()

(313, 3)


,year_sold,month,gas_price
0,1995,1,1.13000
1,1995,2,1.12025
2,1995,3,1.11850
3,1995,4,1.15725
4,1995,5,1.22520


In [8]:
temp = pd.read_csv('../data/raw/archive/temperature.csv')

# Fix datetime
temp['datetime'] = pd.to_datetime(temp['datetime'])
temp['month'] = temp['datetime'].dt.month
temp['year_sold'] = temp['datetime'].dt.year

# Use only a major US city — let's pick Los Angeles (matches CA-heavy sales data)
temp_clean = temp[['datetime','month','year_sold','Los Angeles']].rename(
    columns={'Los Angeles': 'avg_temp'}
)

# Monthly average temperature
temp_monthly = temp_clean.groupby(['year_sold','month'])['avg_temp'].mean().reset_index()

print(temp_monthly.shape)
temp_monthly.head()

(62, 3)


,year_sold,month,avg_temp
0,2012,10,293.255154
1,2012,11,289.276347
2,2012,12,286.239209
3,2013,1,284.742942
4,2013,2,285.734333


In [9]:
# Merge sales with fuel
df = sales_clean.merge(fuel_monthly, on=['year_sold','month'], how='left')

# Merge with temperature
df = df.merge(temp_monthly, on=['year_sold','month'], how='left')

print("Final dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Final dataset shape: (545604, 21)
Columns: ['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate', 'year_sold', 'month', 'season', 'gas_price', 'avg_temp']


,year,make,model,trim,body,transmission,vin,state,condition,odometer,...,interior,seller,mmr,sellingprice,saledate,year_sold,month,season,gas_price,avg_temp
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,...,black,kia motors america inc,20500.0,21500.0,2014-12-16 04:30:00+00:00,2014.0,12.0,Winter,2.6324,281.675190
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,...,beige,kia motors america inc,20800.0,21500.0,2014-12-16 04:30:00+00:00,2014.0,12.0,Winter,2.6324,281.675190
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,...,black,financial services remarketing (lease),31900.0,30000.0,2015-01-14 20:30:00+00:00,2015.0,1.0,Winter,2.2075,282.676923
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,...,black,volvo na rep/world omni,27500.0,27750.0,2015-01-28 20:30:00+00:00,2015.0,1.0,Winter,2.2075,282.676923
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,...,black,financial services remarketing (lease),66000.0,67000.0,2014-12-18 04:30:00+00:00,2014.0,12.0,Winter,2.6324,281.675190


In [10]:
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv('../data/cleaned/main_dataset.csv', index=False)
print("Saved! ✓")

Saved! ✓
